In [61]:
import joblib
import numpy as np
import pandas as pd
import os

In [62]:
def load_app(filepath):
    """
    Application locale — ~25-40 Hz.
    Colonnes attendues : time (Unix s), angle1_l/r (°H), angle2_l/r (°V),
                         gaze_x/y_left/right (pixels), fix_x, fix_y.
    Résolution écran : 1366×768.
    Blinks/pertes : valeurs aux limites de l'écran (≥1350 ou ≥750 px).
    """
    df = pd.read_csv(filepath)

    df['t_s'] = df['time'] - df['time'].iloc[0]   # secondes relatives

    if 'angle1_l' in df.columns:
        df['x'] = (df['angle1_l'] + df['angle1_r']) / 2   # degrés horizontaux
        df['y'] = (df['angle2_l'] + df['angle2_r']) / 2   # degrés verticaux
        df['x_left'] = df['angle1_l']
        df['y_left'] = df['angle2_l']
        df['x_right'] = df['angle1_r']
        df['y_right'] = df['angle2_r']

    # df['pupil'] = np.nan

    return df[['t_s', 'x', 'y', 'x_left', 'y_left', 'x_right', 'y_right']].reset_index(drop=True)


In [63]:
model = joblib.load('randomforest.pkl')

df = load_app('../eyes-tracking/exports/gaze_log(10).csv')
df

/Users/maelle/.pyenv/versions/3.10.6/envs/dyslexia/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/maelle/.pyenv/versions/3.10.6/envs/dyslexia/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/maelle/.pyenv/versions/3.10.6/envs/dyslexia/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarnin

,t_s,x,y,x_left,y_left,x_right,y_right
0,0.000000,8.356829,0.522119,2.792395,-3.835702,13.921263,4.879940
1,0.086368,2.759962,-1.180819,-1.664264,-1.692909,7.184188,-0.668729
2,0.320995,6.057899,-1.747297,1.216114,-2.437552,10.899684,-1.057043
3,0.500193,2.008731,-6.404966,-2.733023,-9.677851,6.750485,-3.132080
4,0.659055,-1.828344,-2.840513,-5.569158,-2.207166,1.912471,-3.473860
...,...,...,...,...,...,...,...
139,25.148793,12.232751,-9.013460,11.952560,-16.708231,12.512941,-1.318689
140,25.315721,12.577418,-10.755835,10.059770,-19.049859,15.095066,-2.461810
141,25.479736,16.268789,-9.996887,14.966881,-18.210600,17.570697,-1.783174
142,25.639775,14.547110,-14.200028,12.963979,-22.044302,16.130241,-6.355755


In [64]:
def extract_features(df, subject_id=None, label=None, source=None):
    """
    Calcule les 9 features comportementales à partir d'un DataFrame normalisé
    (colonnes : t_s, x, y).

    Toutes les features sont des ratios ou coefficients sans unité,
    directement comparables entre DS1, DS2 et l'application.

    Paramètres
    ----------
    df         : DataFrame retourné par load_dataset1/2/app
    subject_id : identifiant du sujet (optionnel)
    label      : 1 = dyslexique, 0 = contrôle, None = inconnu
    source     : 'ds1', 'ds2' ou 'app'

    Retourne
    --------
    dict avec les features + métadonnées (subject_id, dyslexia, source,
    duration_s, n_samples) ou None si données insuffisantes.
    """
    x = df['x'].values
    y = df['y'].values
    t = df['t_s'].values

    if len(x) < 10:
        return None

    # ── Vélocité instantanée ──────────────────────────────────
    dx = np.diff(x)
    dy = np.diff(y)
    dt = np.diff(t)
    dt = np.where(dt < 1e-6, 1e-6, dt)
    raw_vel = np.sqrt(dx**2 + dy**2) / dt

    # Clipper les artefacts (>99.5e percentile = transitions inter-textes, clignements résiduels)
    vel_ceiling = np.percentile(raw_vel, 99.5)
    velocity = np.clip(raw_vel, 0, vel_ceiling)

    # ── Saccades vs fixations ─────────────────────────────────
    # Seuil adaptatif : 75e percentile de la session (robuste aux différences d'unité)
    v_thresh    = np.percentile(velocity, 75)
    is_saccade  = velocity > v_thresh
    is_fixation = ~is_saccade

    # ── Directions ───────────────────────────────────────────
    regression_mask = (dx < 0) & is_saccade   # vers la gauche
    forward_mask    = (dx > 0) & is_saccade   # vers la droite

    n_saccades   = int(np.sum(is_saccade))
    n_regression = int(np.sum(regression_mask))
    n_forward    = int(np.sum(forward_mask))

    fwd_amp = np.mean(np.abs(dx[forward_mask]))    if n_forward    > 0 else 1e-9
    reg_amp = np.mean(np.abs(dx[regression_mask])) if n_regression > 0 else 0.0

    vel_mean   = float(np.mean(velocity))
    vel_std    = float(np.std(velocity))

    # ── Irrégularité des saccades ─────────────────────────────
    saccade_idx = np.where(is_saccade)[0]
    if len(saccade_idx) > 2:
        ipi = np.diff(saccade_idx)   # inter-pulse intervals
        saccade_reg = float(np.std(ipi) / max(np.mean(ipi), 1))
    else:
        saccade_reg = 0.0

    # ── Dérive verticale ──────────────────────────────────────
    mid = len(y) // 2
    y_range = float(np.ptp(y))
    y_drift = float((np.mean(y[mid:]) - np.mean(y[:mid])) / (y_range + 1e-9))

    # ── Assemblage ────────────────────────────────────────────
    feat = {
        'saccade_rate':       float(np.mean(is_saccade)),
        'fixation_prop':      float(np.mean(is_fixation)),
        'regression_rate':    float(n_regression / max(n_saccades, 1)),
        'reg_fwd_ratio':      float(reg_amp / fwd_amp),
        'vel_cv':             float(vel_std / vel_mean if vel_mean > 0 else 0),
        'x_spread_ratio':     float(np.std(x) / (np.ptp(x) + 1e-9)),
        'y_spread_ratio':     float(np.std(y) / (y_range + 1e-9)),
        'y_drift':            y_drift,
        'saccade_regularity': saccade_reg,
        # Métadonnées
        'duration_s':  float(t[-1] - t[0]),
        'n_samples':   len(x),
    }

    if subject_id is not None: feat['subject_id'] = subject_id
    if label      is not None: feat['dyslexia']   = label
    if source     is not None: feat['source']      = source

    return feat

In [65]:
feat = extract_features(df)
feat

{'saccade_rate': 0.2517482517482518,
 'fixation_prop': 0.7482517482517482,
 'regression_rate': 0.4722222222222222,
 'reg_fwd_ratio': 1.7842721986145398,
 'vel_cv': 0.6252391783440369,
 'x_spread_ratio': 0.1703798256454482,
 'y_spread_ratio': 0.2072461719902673,
 'y_drift': 0.08874573875068606,
 'saccade_regularity': 0.7838082767600566,
 'duration_s': 25.81381368637085,
 'n_samples': 144}

In [66]:
feat_df = pd.DataFrame([feat])
feat_df

,saccade_rate,fixation_prop,regression_rate,reg_fwd_ratio,vel_cv,x_spread_ratio,y_spread_ratio,y_drift,saccade_regularity,duration_s,n_samples
0,0.251748,0.748252,0.472222,1.784272,0.625239,0.17038,0.207246,0.088746,0.783808,25.813814,144


In [67]:
FEATURES = ['regression_rate', 'reg_fwd_ratio', 'vel_cv',
            'x_spread_ratio', 'y_spread_ratio', 'y_drift', 'saccade_regularity']

In [68]:
x = np.array([[feat[f] for f in FEATURES]])
x

array([[0.47222222, 1.7842722 , 0.62523918, 0.17037983, 0.20724617,
        0.08874574, 0.78380828]])

In [69]:
prob = model.predict_proba(x)[0]
pred = int(model.predict(x)[0])
prob, pred

(array([0.61691984, 0.38308016]), 0)